In [1]:
# 06_visualization.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 06. Visualization
#
# Publication-ready grayscale figures for the GGI results and sensitivity analysis.
# - seaborn, grayscale palette, no titles/captions (captions handled in the manuscript)
# - every figure saved as both PNG and PDF at 600 dpi into results/figures
#
# Figures:
#   fig1_ggi_heatmap        — GGI by dataset x regulation
#   fig2_component_stack     — component-gap decomposition per dataset (KR baseline)
#   fig3_domain_bars         — mean GGI by domain x regulation
#   fig4_bootstrap_ci        — GGI with bootstrap 95% CI per dataset (KR baseline)
#   fig5_sensitivity_expl    — GGI domain means vs explainability threshold

# %%
# Imports and global style
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Grayscale, print-oriented style
sns.set_theme(style="white", context="paper")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "savefig.bbox": "tight",
    "figure.dpi": 150,          # on-screen; export dpi set on save
})

DPI = 600
REG_ORDER = ["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]
REG_LABELS = {"KR_AI_Basic_Act": "KR AI Basic Act", "EU_AI_Act": "EU AI Act", "NAIC": "NAIC"}

# Save a figure as both PNG and PDF at 600 dpi (no title added here)
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=DPI)
    plt.close(fig)

# %%
# Load result tables produced by notebooks 04 and 05
ggi_full = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"))
comp_scores = pd.read_csv(os.path.join(TAB_DIR, "04_component_scores.csv"))
boot = pd.read_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"))
sens_expl = pd.read_csv(os.path.join(TAB_DIR, "05_sensitivity_explainability.csv"))

DATASET_ORDER = ["German Credit", "Credit Card Approval", "Medical Cost", "Porto Seguro"]

# %%
# Figure 1: GGI heatmap (dataset x regulation), grayscale
mat = (ggi_full.pivot(index="dataset", columns="regulation", values="GGI")
       .reindex(index=DATASET_ORDER, columns=REG_ORDER))

fig, ax = plt.subplots(figsize=(5.2, 4.0))
sns.heatmap(
    mat, ax=ax, cmap="Greys", annot=True, fmt=".3f",
    linewidths=0.6, linecolor="black", cbar_kws={"label": "GGI"},
    annot_kws={"color": "black"},
)
# Force annotation contrast: high cells get white text
for text, val in zip(ax.texts, mat.values.flatten()):
    text.set_color("white" if val > mat.values.max() * 0.6 else "black")
ax.set_xticklabels([REG_LABELS[c] for c in REG_ORDER], rotation=20, ha="right")
ax.set_ylabel("")
ax.set_xlabel("")
save_fig(fig, "fig1_ggi_heatmap")

# %%
# Figure 2: component-gap decomposition per dataset (stacked bars), KR baseline
# Uses hatches (not color) to distinguish components in grayscale.
kr = ggi_full[ggi_full["regulation"] == "KR_AI_Basic_Act"].set_index("dataset")
kr = kr.reindex(DATASET_ORDER)
comp_cols = ["g_PERF", "g_DI", "g_SP", "g_EXPL"]
comp_labels = ["Performance", "Disparate Impact", "Statistical Parity", "Explainability"]
grays = ["0.15", "0.40", "0.65", "0.85"]
hatches = ["", "//", "..", "xx"]

fig, ax = plt.subplots(figsize=(6.0, 4.0))
bottom = np.zeros(len(DATASET_ORDER))
x = np.arange(len(DATASET_ORDER))
for col, lab, g, h in zip(comp_cols, comp_labels, grays, hatches):
    vals = kr[col].fillna(0).values
    ax.bar(x, vals, bottom=bottom, width=0.6, label=lab,
           color=g, edgecolor="black", linewidth=0.7, hatch=h)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=20, ha="right")
ax.set_ylabel("Component gap contribution")
ax.legend(frameon=False, fontsize=9, loc="upper left")
sns.despine(ax=ax)
save_fig(fig, "fig2_component_stack")

# %%
# Figure 3: mean GGI by domain x regulation (grouped bars), grayscale
dom = (ggi_full.groupby(["domain", "regulation"])["GGI"].mean()
       .reset_index()
       .pivot(index="domain", columns="regulation", values="GGI")
       .reindex(columns=REG_ORDER))
dom = dom.reindex(["credit", "insurance"])

fig, ax = plt.subplots(figsize=(6.0, 4.0))
x = np.arange(len(dom.index))
width = 0.25
reg_grays = {"KR_AI_Basic_Act": "0.25", "EU_AI_Act": "0.55", "NAIC": "0.80"}
for i, reg in enumerate(REG_ORDER):
    ax.bar(x + (i - 1) * width, dom[reg].values, width,
           label=REG_LABELS[reg], color=reg_grays[reg],
           edgecolor="black", linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in dom.index])
ax.set_ylabel("Mean GGI")
ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig3_domain_bars")

# %%
# Figure 4: GGI with bootstrap 95% CI per dataset (KR baseline), grayscale
b = boot[boot["regulation"] == "KR_AI_Basic_Act"].set_index("dataset").reindex(DATASET_ORDER)
means = b["GGI_mean"].values
lo = means - b["CI_low"].values
hi = b["CI_high"].values - means

fig, ax = plt.subplots(figsize=(6.0, 4.0))
x = np.arange(len(DATASET_ORDER))
# marker shape encodes domain (grayscale-friendly)
domain_marker = {"credit": "o", "insurance": "s"}
for xi, ds in zip(x, DATASET_ORDER):
    dm = b.loc[ds, "domain"]
    ax.errorbar(xi, b.loc[ds, "GGI_mean"],
                yerr=[[means[xi] - b.loc[ds, "CI_low"]], [b.loc[ds, "CI_high"] - means[xi]]],
                fmt=domain_marker[dm], color="black", ecolor="0.4",
                elinewidth=1.2, capsize=4, markersize=7,
                markerfacecolor="white" if dm == "insurance" else "black")
ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=20, ha="right")
ax.set_ylabel("GGI (KR AI Basic Act)")
# legend proxies for domain markers
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker="o", color="black", markerfacecolor="black",
           linestyle="", markersize=7, label="Credit"),
    Line2D([0], [0], marker="s", color="black", markerfacecolor="white",
           linestyle="", markersize=7, label="Insurance"),
]
ax.legend(handles=legend_elems, frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig4_bootstrap_ci")

# %%
# Figure 5: sensitivity of domain-mean GGI to the explainability threshold, grayscale
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.plot(sens_expl["tau_EXPL"], sens_expl["credit_mean"],
        marker="o", color="black", linewidth=1.5, label="Credit")
ax.plot(sens_expl["tau_EXPL"], sens_expl["insurance_mean"],
        marker="s", color="0.5", linewidth=1.5, linestyle="--",
        markerfacecolor="white", label="Insurance")
ax.set_xlabel(r"Explainability threshold $\tau_{\mathrm{EXPL}}$")
ax.set_ylabel("Mean GGI")
ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig5_sensitivity_expl")

# %%
# List saved figures
saved = sorted(f for f in os.listdir(FIG_DIR) if f.endswith((".png", ".pdf")))
for f in saved:
    print(f)

fig1_ggi_heatmap.pdf
fig1_ggi_heatmap.png
fig2_component_stack.pdf
fig2_component_stack.png
fig3_domain_bars.pdf
fig3_domain_bars.png
fig4_bootstrap_ci.pdf
fig4_bootstrap_ci.png
fig5_sensitivity_expl.pdf
fig5_sensitivity_expl.png
